In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import sys
import logging
import os
import random
import time
import re
import math
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
from dateutil import parser as date_parser

# --- Selenium ---
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

# --- Requests ---
import requests
import warnings
from urllib3.exceptions import InsecureRequestWarning

warnings.simplefilter('ignore', InsecureRequestWarning)

# ===============================================================
#  1. 全域設定
# ===============================================================

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

DATA_FILENAME = os.path.join(BASE_DIR, 'News_Data.csv')
LOG_FILENAME = os.path.join(BASE_DIR, 'scraper_final.log')
V5_COLUMNS = ['url', 'media', 'category', 'publish_time', 'title', 'content', 'fetch_time', 'label']

LIMIT_L1 = 50
LIMIT_L0 = 75

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILENAME, encoding='utf-8', mode='a'),
        logging.StreamHandler(sys.stdout)
    ]
)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Referer': 'https://www.google.com/'
}

# ===============================================================
#  2. 媒體設定檔
# ===============================================================

MEDIA_CONFIGS = [
    {'name': 'SETN', 'hot_url': 'https://www.setn.com/ViewAll.aspx?PageGroupID=6', 'latest_url': 'https://www.setn.com/ViewAll.aspx', 'base_url': 'https://www.setn.com', 'hot_selector': ['h3.view-li-title a', 'div.news-list-group a.gt', 'div.col-sm-12 h3 a'], 'latest_selector': ['h3.view-li-title a', 'div.col-sm-12 > a']},
    {'name': 'TVBS', 'hot_url': 'https://news.tvbs.com.tw/hot', 'latest_url': 'https://news.tvbs.com.tw/realtime', 'base_url': 'https://news.tvbs.com.tw', 'hot_selector': ['div.news_list li a', 'div.list li a'], 'latest_selector': ['div.news_list li a']},
    {'name': 'ETtoday', 'hot_url': 'https://www.ettoday.net/news/hot-news.htm', 'latest_url': 'https://www.ettoday.net/news/news-list.htm', 'base_url': 'https://www.ettoday.net', 'hot_selector': ['div.part_list_2 h3 a', 'div.block_content h3 a'], 'latest_selector': ['div.part_list_2 h3 a']},
    {'name': 'LTN', 'hot_url': 'https://news.ltn.com.tw/list/breakingnews/popular', 'latest_url': 'https://news.ltn.com.tw/list/breakingnews', 'base_url': 'https://news.ltn.com.tw', 'hot_selector': ['div.whitecon a', 'ul.list a', 'div.boxTitle a', 'a.tit'], 'latest_selector': ['div.whitecon ul.list li a', 'div.whitecon a', 'ul.list a']},
    {'name': 'ChinaTimes', 'hot_url': 'https://www.chinatimes.com/hotnews/?chdtv', 'latest_url': 'https://www.chinatimes.com/realtimenews/?chdtv', 'base_url': 'https://www.chinatimes.com', 'hot_selector': ['section.hot-news li h3.title a', 'ul.vertical-list li h3 a'], 'latest_selector': ['section.article-list li h3.title a']},
    {'name': 'UDN', 'hot_url': 'https://udn.com/rank/pv/2', 'latest_url': 'https://udn.com/news/breaknews/1', 'base_url': 'https://udn.com', 'hot_selector': ['div.story-list__text h2 a', 'section.rank-list h3 a'], 'latest_selector': ['div.story-list__text h2 a']}
]

CONTENT_SELECTORS = [
    'div.article-content', 'div.story', 'div.article-body', 'div[itemprop="articleBody"]', 
    'div.text', 'div.news_content', 'div.content', 'section.article-content__editor',
    'div.p-note', 'div.fncnews-content', 'div.post-content', 'article'
]

# ===============================================================
#  3. 工具模組
# ===============================================================

def clean_category(cat_str):
    if not cat_str or cat_str == "即時": return "即時"
    if '>' in cat_str: cat_str = cat_str.split('>')[-1]
    return cat_str.strip()

def format_time_strict(time_str):
    now = datetime.now()
    if not time_str or str(time_str).lower() in ['nan', 'n/a', 'none', '']:
        return now.strftime("%Y/%-m/%-d %-I:%M:%S %p")
    try:
        raw = str(time_str).replace("發布時間：", "").replace("更新時間：", "").strip()
        if "小時前" in raw:
            hours = int(re.search(r'(\d+)', raw).group(1))
            dt = now - timedelta(hours=hours)
        elif "分鐘前" in raw:
            mins = int(re.search(r'(\d+)', raw).group(1))
            dt = now - timedelta(minutes=mins)
        elif "剛剛" in raw:
            dt = now
        else:
            dt = date_parser.parse(raw, fuzzy=True)
        return f"{dt.year}/{dt.month}/{dt.day} {int(dt.strftime('%I'))}:{dt.strftime('%M:%S %p')}"
    except:
        return now.strftime("%Y/%-m/%-d %-I:%M:%S %p")

def parse_html_content(html_text):
    soup = BeautifulSoup(html_text, 'lxml')
    content = np.nan
    for sel in CONTENT_SELECTORS:
        elm = soup.select_one(sel)
        if elm:
            for tag in elm(['script', 'style', 'iframe', 'div.knn_related', 'div.fb-root', 'div.author', 'div.ad']):
                tag.decompose()
            text = elm.get_text(separator=' ', strip=True)
            text = re.sub(r'\s+', ' ', text)
            if len(text) > 30:
                content = text
                break
    category = "即時"
    meta_sec = soup.select_one('meta[property="article:section"]')
    if meta_sec and meta_sec.get('content'): category = meta_sec.get('content')
    raw_time = None
    time_elm = soup.select_one('meta[property="article:published_time"], meta[name="pubdate"], time, span.time, div.date, div.meta-info time')
    if time_elm: raw_time = time_elm.get('content') if time_elm.has_attr('content') else time_elm.get_text(strip=True)
    return clean_category(category), format_time_strict(raw_time), content

# ===============================================================
#  4. 爬蟲核心 (V12.0)
# ===============================================================

class HumanNavigator:
    def __init__(self, driver):
        self.driver = driver
        self.action = ActionChains(driver)

    def random_sleep(self, min_s=1.5, max_s=3.0):
        time.sleep(random.uniform(min_s, max_s))

    def smooth_scroll(self):
        try:
            current_pos = self.driver.execute_script("return window.pageYOffset;")
            target_distance = random.randint(800, 1200)
            steps = random.randint(10, 20)
            for _ in range(steps):
                move = (target_distance / steps) + random.uniform(-5, 5) 
                current_pos += move
                self.driver.execute_script(f"window.scrollTo(0, {current_pos});")
                time.sleep(0.02)
        except: pass

class NewsSpider:
    def __init__(self):
        self.options = Options()
        self.options.add_argument("--headless")
        self.options.add_argument("--disable-gpu")
        self.options.add_argument("--no-sandbox")
        self.options.add_argument("--window-size=1920,1080")
        self.options.add_argument(f"user-agent={HEADERS['User-Agent']}")
        self.options.add_argument("--disable-blink-features=AutomationControlled")
        self.options.add_experimental_option("excludeSwitches", ["enable-automation"])
        self.options.add_experimental_option('useAutomationExtension', False)
        prefs = {"profile.managed_default_content_settings.images": 2}
        self.options.add_experimental_option("prefs", prefs)
        self.driver = None
        self.human = None
        # [V12.0] 改為 Dictionary {url: label} 以便查詢舊標籤
        self.existing_data = self._get_existing_data()

    def _get_existing_data(self):
        """讀取現有資料庫，建立 {url: label} 對照表"""
        existing = {}
        if os.path.exists(DATA_FILENAME):
            try:
                df = pd.read_csv(DATA_FILENAME)
                if 'url' in df.columns and 'label' in df.columns:
                    for idx, row in df.iterrows():
                        existing[str(row['url'])] = row['label']
            except: pass
        return existing

    def start(self):
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=self.options)
        self.driver.set_page_load_timeout(45)
        self.driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        self.human = HumanNavigator(self.driver)

    def stop(self):
        if self.driver:
            try: self.driver.quit()
            except: pass
            self.driver = None

    def fetch_content_smart(self, url):
        # Requests
        try:
            res = requests.get(url, headers=HEADERS, timeout=6, verify=False)
            if res.status_code == 200:
                cat, t, c = parse_html_content(res.text)
                if isinstance(c, str): return cat, t, c
        except: pass
        # Selenium
        try:
            self.driver.execute_script("window.open('');")
            self.driver.switch_to.window(self.driver.window_handles[-1])
            
            # [V12.0] Timeout 防禦
            try:
                self.driver.get(url)
            except TimeoutException:
                logging.warning("      ⚠️ 內文載入超時，嘗試停止載入並解析...")
                self.driver.execute_script("window.stop();")
            
            self.human.random_sleep(1.0, 2.0)
            html = self.driver.page_source
            self.driver.close()
            self.driver.switch_to.window(self.driver.window_handles[0])
            return parse_html_content(html)
        except:
            try:
                if len(self.driver.window_handles) > 1:
                    self.driver.close()
                    self.driver.switch_to.window(self.driver.window_handles[0])
            except: pass
        return "即時", format_time_strict(None), np.nan

    def crawl_task(self, site_config, label, limit):
        url = site_config['hot_url'] if label == 1 else site_config['latest_url']
        selectors = site_config['hot_selector'] if label == 1 else site_config['latest_selector']
        site_name = site_config['name']

        logging.info(f"      L{label} 任務啟動 (目標: {limit} 筆)...")
        try:
            # [V12.0] 列表頁載入 Timeout 防禦
            try:
                self.driver.get(url)
                try:
                    WebDriverWait(self.driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
                    if site_name == 'LTN': time.sleep(5) 
                    else: self.human.random_sleep(2, 4)
                except: pass
            except TimeoutException:
                logging.warning("      ⚠️ 列表載入超時，嘗試強制停止並解析已載入內容...")
                self.driver.execute_script("window.stop();")

            collected = []
            seen_links = set()
            
            # 捲動
            scroll_target = int((limit / 8) * 1.5) + 3
            for _ in range(scroll_target):
                self.human.smooth_scroll()
                self.human.random_sleep(1.0, 1.5)
                soup = BeautifulSoup(self.driver.page_source, 'lxml')
                found_count = 0
                for sel in selectors: found_count += len(soup.select(sel))
                if found_count >= limit * 1.2: break
            
            soup = BeautifulSoup(self.driver.page_source, 'lxml')
            elements = []
            for sel in selectors:
                found = soup.select(sel)
                if found: elements.extend(found)

            logging.info(f"      -> 掃描到 {len(elements)} 個連結，開始處理...")

            for item in elements:
                if len(collected) >= limit: break
                try:
                    link = item.get('href')
                    if site_name == 'LTN':
                        if not link: continue
                        if not any(x in link for x in ['/news/', '/article/', 'sports', 'ent', 'ec', 'auto']): continue
                    
                    title = item.get('title') or item.get_text(strip=True)
                    if not link or not title or len(title) < 4: continue
                    
                    if link.startswith('/'):
                        base = site_config['base_url']
                        full_url = base.rstrip('/') + '/' + link.lstrip('/')
                    elif link.startswith('http'):
                        full_url = link
                    else: continue
                    
                    # URL 處理
                    if 'setn.com' in full_url and 'NewsID=' in full_url: clean_url = full_url 
                    elif 'udn.com' in full_url and 'story/' in full_url: clean_url = full_url 
                    else: clean_url = full_url.split('?')[0]

                    if clean_url in seen_links: continue
                    seen_links.add(clean_url)

                    # [V12.0] 智慧去重與標籤升級邏輯
                    # 1. 檢查資料庫是否已有此網址
                    is_in_db = clean_url in self.existing_data
                    db_label = self.existing_data.get(clean_url, -1)

                    if is_in_db:
                        # 情況 A: 現在抓到 L1，資料庫是 L0 -> 升級！ (抓取並覆蓋)
                        if label == 1 and db_label == 0:
                            logging.info(f"         [Upgrade] 發現 L0->L1 潛力股: {title[:8]}")
                            # 繼續執行抓取，後續寫入時會變成新的一筆 L1 (舊的 L0 仍在，可做對照)
                        
                        # 情況 B: 現在抓到 L1，資料庫也是 L1 -> 重複，跳過
                        elif label == 1 and db_label == 1:
                            continue
                        
                        # 情況 C: 現在抓到 L0，資料庫已有 (無論 L0/L1) -> 重複，跳過
                        elif label == 0:
                            continue
                    
                    # 抓取流程
                    cat, t_str, content = self.fetch_content_smart(full_url)
                    if not isinstance(content, str) or len(content) < 30: continue 

                    fetch_time = datetime.now().strftime("%Y/%-m/%-d %-I:%M:%S %p")
                    collected.append({
                        'url': full_url, # 存完整網址
                        'media': site_name,
                        'category': cat,
                        'publish_time': t_str,
                        'title': title,
                        'content': content,
                        'fetch_time': fetch_time,
                        'label': label
                    })
                    print(f"         [L{label}] {title[:8]}... (OK)")
                    time.sleep(random.uniform(1.5, 3.0))

                except Exception: continue
            
            return collected

        except Exception as e:
            logging.error(f"      🔥 任務錯誤: {e}")
            return []

# ===============================================================
#  5. 主程式
# ===============================================================

def main():
    logging.info(f"🚀 T-Forecast V12.0 (Label Upgrade & Timeout Fix) 啟動")
    
    spider = NewsSpider()
    
    for site_config in MEDIA_CONFIGS:
        site_name = site_config['name']
        logging.info(f"--- 處理媒體: {site_name} ---")
        spider.start()
        all_data = []
        try:
            l1_data = spider.crawl_task(site_config, label=1, limit=LIMIT_L1)
            all_data.extend(l1_data)
            logging.info(f"      ✅ {site_name} L1 完成: {len(l1_data)} 筆")
            spider.human.random_sleep(2, 3)
            
            l0_data = spider.crawl_task(site_config, label=0, limit=LIMIT_L0)
            all_data.extend(l0_data)
            logging.info(f"      ✅ {site_name} L0 完成: {len(l0_data)} 筆")
        except Exception as e:
            logging.error(f"   🔥 {site_name} 發生錯誤: {e}")
        
        spider.stop()
        logging.info(f"   💤 休息 5 秒...")
        time.sleep(5)

        if all_data:
            df = pd.DataFrame(all_data)
            for col in V5_COLUMNS:
                if col not in df.columns: df[col] = np.nan
            df = df[V5_COLUMNS]
            try:
                df.to_csv(DATA_FILENAME, mode='a', header=False, index=False, encoding='utf-8-sig')
                logging.info(f"   💾 {site_name} 存檔成功 (共 {len(all_data)} 筆)")
            except:
                df.to_csv(f"backup_{site_name}_{int(time.time())}.csv", index=False)

    logging.info("🏁 任務結束")

if __name__ == "__main__":
    main()

2025-12-10 11:15:03,472 - INFO - 🚀 T-Forecast V12.0 (Label Upgrade & Timeout Fix) 啟動
2025-12-10 11:15:03,541 - INFO - --- 處理媒體: SETN ---
2025-12-10 11:15:03,541 - INFO - ====== WebDriver manager ======
2025-12-10 11:15:03,706 - INFO - Get LATEST chromedriver version for google-chrome
2025-12-10 11:15:03,920 - INFO - Get LATEST chromedriver version for google-chrome
2025-12-10 11:15:04,168 - INFO - Get LATEST chromedriver version for google-chrome
2025-12-10 11:15:04,854 - INFO - WebDriver version 143.0.7499.40 selected
2025-12-10 11:15:04,858 - INFO - Modern chrome version https://storage.googleapis.com/chrome-for-testing-public/143.0.7499.40/mac-arm64/chromedriver-mac-arm64.zip
2025-12-10 11:15:04,858 - INFO - About to download new driver from https://storage.googleapis.com/chrome-for-testing-public/143.0.7499.40/mac-arm64/chromedriver-mac-arm64.zip
2025-12-10 11:15:04,925 - INFO - Driver downloading response is 200
2025-12-10 11:15:05,271 - INFO - Get LATEST chromedriver version for 